# 04 — Modeling

Can information available **before the day-ahead auction** (10:30 EPT on D-1) predict the DA/RT spread?

| Model | Target | Question |
|---|---|---|
| Direction | 1 if RT > DA | Can we call the sign? (plan baseline: logistic regression) |
| Magnitude | spread, fit clipped to ±$100 | Can we predict the dollar value? (OLS / ridge) |
| Risk | log(1 + \|spread\|) | Can we predict *how big* the spread will be, regardless of sign? |

**Protocol** — chronological only, no random splits:
* hyperparameters chosen on **expanding-window walk-forward folds inside the training period** (train on everything before each quarter of 2024, evaluate on that quarter);
* final models fit on **2023–2024**, compared on **validation = 2025 H1**;
* **test (2025 H2) and holdout (2026) stay sealed** — the test period is evaluated once, in the Phase 6 backtest;
* every model is compared against a constant baseline fitted on the training period;
* inputs: the 31 tradable features (Phase 3), winsorized at the training 1st/99th percentiles (handles Winter Storm Elliott in early-2023 trailing windows); hours without a published D-1 load forecast get no prediction.

In [ ]:
import sys, pathlib, warnings
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, log_loss, mean_absolute_error
from src.config import load_config
from src.analysis import plots as pl
from src.models.split import usable, walk_forward_folds
from src.models.common import feature_spec, to_model_frame, grouped_permutation_importance
from src.models import classifier as clf, baseline as bl
from src.models.selected import CHOICES, fit_selected, predict_selected

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.4f}".format)
pl.apply_style()
cfg = load_config(); FIG = cfg["paths"]["figures"]
NOTE = "PJM Data Miner 2. Models fit on 2023–2024; evaluated on 2025 H1 (validation). Test and holdout untouched."

F = pd.read_parquet(cfg["paths"]["processed"] / "features.parquet")      # holdout not in this file
spec = feature_spec()
ok = usable(F, spec.columns)
D = F[ok & F["period"].isin(["train", "validation"])].copy()             # test rows are not loaded into D
tr, va = D[D.period == "train"], D[D.period == "validation"]
Xtr, Xva = to_model_frame(tr, spec), to_model_frame(va, spec)
ytr, yva = tr["target_rt_above_da"].to_numpy(), va["target_rt_above_da"].to_numpy()
s_tr, s_va = tr["da_rt_spread"].to_numpy(), va["da_rt_spread"].to_numpy()
folds = walk_forward_folds(D)
print(f"{len(spec.columns)} model inputs ({len(spec.numeric)} numeric, {len(spec.categorical)} one-hot, {len(spec.binary)} binary)")
print(f"usable rows: train {len(tr):,} / {(F.period == 'train').sum():,}, validation {len(va):,} / {(F.period == 'validation').sum():,}")
print("walk-forward folds:", [(name, len(a), len(b)) for a, b, name in folds])

## 1. Hyperparameters on walk-forward folds (training data only)

In [ ]:
def walk_forward(make, kind, **kw):
    out = []
    for a, b, name in folds:
        Xa, Xb = to_model_frame(D.loc[a], spec), to_model_frame(D.loc[b], spec)
        sa, sb = D.loc[a, "da_rt_spread"].to_numpy(), D.loc[b, "da_rt_spread"].to_numpy()
        ya, yb = D.loc[a, "target_rt_above_da"].to_numpy(), D.loc[b, "target_rt_above_da"].to_numpy()
        if kind == "direction":
            p = make(spec, **kw).fit(Xa, ya).predict_proba(Xb)[:, 1]
            out.append({"fold": name, "auc": roc_auc_score(yb, p), "log_loss": log_loss(yb, np.clip(p, 1e-6, 1 - 1e-6)),
                        "const_log_loss": log_loss(yb, np.full(len(yb), ya.mean()), labels=[0, 1])})
        elif kind == "magnitude":
            p = make(spec, **kw).fit(Xa, bl.clip_target(sa)).predict(Xb)
            out.append({"fold": name, "mae": mean_absolute_error(sb, p), "const_mae": mean_absolute_error(sb, np.full(len(sb), np.median(sa)))})
        else:
            p = make(spec, **kw).fit(Xa, bl.risk_target(sa)).predict(Xb)
            out.append({"fold": name, "spearman": pd.Series(p).corr(pd.Series(np.abs(sb)), method="spearman")})
    return pd.DataFrame(out)

rows = []
for C in (0.001, 0.01, 0.1, 1.0):
    r = walk_forward(clf.make_logit, "direction", C=C)
    rows.append({"model": f"logistic C={C}", "mean_auc": r.auc.mean(), "mean_log_loss": r.log_loss.mean(), "const_log_loss": r.const_log_loss.mean(), "fold_aucs": r.auc.round(3).tolist()})
for lr, it in ((0.05, 150), (0.03, 300)):
    r = walk_forward(clf.make_gbm, "direction", learning_rate=lr, max_iter=it)
    rows.append({"model": f"gbm lr={lr} iter={it}", "mean_auc": r.auc.mean(), "mean_log_loss": r.log_loss.mean(), "const_log_loss": r.const_log_loss.mean(), "fold_aucs": r.auc.round(3).tolist()})
pd.DataFrame(rows)

In [ ]:
rows = []
for a in (10, 100, 1000, 10000):
    r = walk_forward(bl.make_ridge, "magnitude", alpha=a)
    rows.append({"model": f"ridge alpha={a}", "mean_mae": r.mae.mean(), "const_median_mae": r.const_mae.mean(), "fold_maes": r.mae.round(2).tolist()})
r = walk_forward(bl.make_ols, "magnitude")
rows.append({"model": "ols", "mean_mae": r.mae.mean(), "const_median_mae": r.const_mae.mean(), "fold_maes": r.mae.round(2).tolist()})
display(pd.DataFrame(rows))
rows = []
for a in (10, 100, 1000):
    r = walk_forward(bl.make_ridge, "risk", alpha=a)
    rows.append({"model": f"risk ridge alpha={a}", "mean_spearman": r.spearman.mean(), "fold_spearman": r.spearman.round(3).tolist()})
risk_wf = walk_forward(bl.make_ridge, "risk", alpha=CHOICES["risk"]["alpha"])
dir_wf = walk_forward(clf.make_logit, "direction", C=CHOICES["direction"]["C"])
pd.DataFrame(rows)

In [ ]:
# Pre-specified check: is the weak direction result an artefact of the feature set? (training folds only)
rows = []
for label, excl in (("all features", ()), ("no calendar", ("calendar",)), ("fundamental + forecast bias", ("calendar", "market"))):
    s2 = feature_spec(exclude_groups=excl)
    aucs = []
    for a, b, name in folds:
        m = clf.make_logit(s2, C=CHOICES["direction"]["C"]).fit(to_model_frame(D.loc[a], s2), D.loc[a, "target_rt_above_da"])
        aucs.append(roc_auc_score(D.loc[b, "target_rt_above_da"], m.predict_proba(to_model_frame(D.loc[b], s2))[:, 1]))
    rows.append({"feature set": label, "mean_auc": np.mean(aucs), "fold_aucs": np.round(aucs, 3).tolist()})
pd.DataFrame(rows)

In [ ]:
print("Selected (on walk-forward folds):", CHOICES)
models = fit_selected(tr, spec)
pred = predict_selected(models, va, spec)
gbm = clf.make_gbm(spec, learning_rate=0.03, max_iter=300).fit(Xtr, ytr)

## 2. Direction model on validation (2025 H1)

In [ ]:
pd.DataFrame({
    "logistic (selected)": clf.classification_metrics(yva, pred["p_rt_above_da"], s_va, base_rate=ytr.mean()),
    "gradient boosting": clf.classification_metrics(yva, gbm.predict_proba(Xva)[:, 1], s_va, base_rate=ytr.mean()),
})

In [ ]:
rel = clf.reliability_table(yva, pred["p_rt_above_da"])
pl.reliability_plot(rel, "Direction probabilities carry almost no information",
    "Logistic model, validation 2025 H1: observed share of RT > DA hours by predicted-probability decile.",
    FIG / "09_direction_calibration.png", NOTE, base_rate=yva.mean())
rel

In [ ]:
# Trading only on confident predictions (preview of Phase 5; before costs)
clf.coverage_table(pred["p_rt_above_da"].to_numpy(), s_va)

In [ ]:
clf.logit_coefficients(models["direction"]).head(15)

## 3. Magnitude model on validation

In [ ]:
ols = bl.make_ols(spec).fit(Xtr, bl.clip_target(s_tr))
gbr = bl.make_gbm_regressor(spec).fit(Xtr, bl.clip_target(s_tr))
pd.DataFrame({"ols": bl.regression_metrics(s_va, ols.predict(Xva), s_tr),
              "ridge (selected)": bl.regression_metrics(s_va, pred["pred_spread"], s_tr),
              "gbm (absolute loss)": bl.regression_metrics(s_va, gbr.predict(Xva), s_tr)})

## 4. Risk model on validation

In [ ]:
rho = pd.Series(pred["pred_risk"].to_numpy()).corr(pd.Series(np.abs(s_va)), method="spearman")
print(f"Spearman(predicted risk, |spread|) on validation: {rho:.3f}")
dec = bl.risk_deciles(pred["pred_risk"].to_numpy(), s_va)
pl.risk_decile_bars(dec, "Spread risk is predictable before the auction",
    f"Validation 2025 H1: realized |RT − DA| by decile of predicted risk (ridge on tradable features). Spearman ρ = {rho:.2f}.",
    FIG / "10_risk_deciles.png", NOTE)
dec

In [ ]:
stab = pd.DataFrame({"block": [f"Fold {n}" for *_, n in folds] + ["Validation 2025 H1"],
                     "direction_auc": dir_wf["auc"].tolist() + [roc_auc_score(yva, pred["p_rt_above_da"])],
                     "risk_spearman": risk_wf["spearman"].tolist() + [rho]})
pl.fold_stability(stab, "Risk is predictable in every period; direction is not",
    "Walk-forward folds (2024 quarters, trained on all earlier data) and validation. Reference lines: 0.50 AUC and 0 correlation = no skill.",
    FIG / "11_model_stability.png", NOTE)
stab

## 5. What drives each model? Grouped permutation importance on validation

Each feature group's columns are shuffled together, whole days at a time; importance = drop in the metric.

In [ ]:
block = va["market_date"].reset_index(drop=True)
imp_dir = grouped_permutation_importance(lambda X: models["direction"].predict_proba(X)[:, 1], Xva.reset_index(drop=True), yva,
                                         roc_auc_score, spec.groups, n_repeats=10, block=block)
spear = lambda y, p: pd.Series(p).corr(pd.Series(y), method="spearman")
imp_risk = grouped_permutation_importance(models["risk"].predict, Xva.reset_index(drop=True), np.abs(s_va), spear,
                                          spec.groups, n_repeats=10, block=block)
pl.importance_panels({"Direction model — drop in ROC-AUC": imp_dir, "Risk model — drop in Spearman ρ": imp_risk},
    "Market history and load fundamentals carry the risk signal",
    "Grouped permutation importance on validation 2025 H1 (whole days shuffled; bars = mean ± std over 10 repeats).",
    FIG / "12_feature_importance.png", NOTE)
display(imp_dir); imp_risk

## Summary

**Before the auction, the size of the spread is predictable. Its direction is not.**

| Model (validation 2025 H1) | Result | vs constant baseline |
|---|---|---|
| Direction — logistic (C = 0.01) | ROC-AUC 0.53, accuracy 64.2% | Brier skill **−3%**; never predicts RT > DA, so accuracy just equals "always RT < DA" (64.2%) |
| Direction — gradient boosting | ROC-AUC 0.52 | Brier skill −3% |
| Magnitude — OLS / ridge / GBM | MAE $15.1–15.8 | no skill: constant train median gives MAE $15.04 |
| **Risk — ridge on log(1 + \|spread\|)** | **Spearman ρ = 0.49** | top predicted-risk decile averages **$58** \|spread\| vs **$4** in the bottom decile; 28% of top-decile hours exceed $50 vs 0% |

* **Stable across time:** the risk correlation holds on every walk-forward fold (0.47–0.67) and on validation (0.49). Direction AUC is ≈ 0.50 on three of four folds; only Q1 2024 reached 0.60.
* **Not a feature-set artefact:** three pre-specified direction feature sets all give a walk-forward AUC of 0.52–0.53.
* **What drives risk:** market history (recent DA level, RT and spread volatility) and load fundamentals (forecast level, position vs daily peak). Calendar features add nothing and slightly hurt the direction model.
* **Calibration:** predicted P(RT > DA) sits at 15–39% while the observed rate stays at 32–41% across all deciles. The probabilities barely separate hours. The base rate rose from 32% (2023–24) to 36% (2025 H1).
* **Trading preview (before costs):** trading only confident direction calls means always being *short* the spread. That wins about 64% of hours (median +$3) but loses on average (−$0.4 to −$0.9 per traded hour), because the rare spikes dominate. This repeats the Phase 2 finding out of sample.

**Implications for Phase 5.** The plan's probability-threshold signal can be built (and will be, as the plan's model-based signal), but on this evidence it degenerates into "always short". The useful information is the **risk model**: a candidate signal takes the base-rate short position only in hours with *low predicted risk* and stays flat in high-risk hours, where the spikes live. Thresholds must be chosen on train/validation only, and the test period is still sealed.